<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/Heart-Disease-Prediction/blob/main/gifair_per_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 5: GIFAIR-FL-Per (Yue et al. 2022)

**Algorithm 3 — Yue et al. 2022 (exact):**
Personalised variant of GIFAIR-Global that addresses its main limitation.
GIFAIR-Global needs an extra communication round to compute r_k from global
model losses. GIFAIR-Per circumvents this by computing r_k from each client's
own personalised model losses — no extra round needed.

Key differences from GIFAIR-Global (Algorithm 2):
- r_k computed from PERSONALISED losses F_k(theta_k), not global model losses
- Each client retains its own theta_k after each round (personalised solution)
- Server stores one set of personalised weights per hospital
- Final evaluation uses each hospital's theta_k, not the global model
- Prevents Hospital_B extreme distribution from destabilising global model

Citation: Yue et al. 2022, INFORMS Journal on Data Science.
lambda=0.1

## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 32.6 MB/s eta 0:00:00
✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.1
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, parameters_to_ndarrays,
                          ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr     : {fl.__version__}")
print(f"torch    : {torch.__version__}")
print(f"numpy    : {np.__version__}")
print(f"GPU      : {torch.cuda.is_available()}")
print("✓ All libraries ready")

flwr     : 1.32.1
torch    : 2.11.0+cu128
numpy    : 2.0.2
GPU      : True
✓ All libraries ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device     : {device}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"Memory     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device     : cuda
GPU        : NVIDIA L4
Memory     : 23.7 GB
CUDA       : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

In [ ]:
import os, shutil, time

DRIVE_PATH = '/content/drive/MyDrive/dissertation/archive'
LOCAL_PATH = '/content/nih_unzipped'

if os.path.exists(LOCAL_PATH) and len(os.listdir(LOCAL_PATH)) > 5:
    print(f"✓ Already on local disk — skipping copy")
else:
    print("Copying from Drive to local disk...")
    t0 = time.time()
    shutil.copytree(DRIVE_PATH, LOCAL_PATH)
    print(f"✓ Done in {(time.time()-t0)/60:.1f} minutes")

DATASET_PATH = LOCAL_PATH
print(f"Dataset path: {DATASET_PATH}")

Copying from Drive to local disk...


In [ ]:
'''
import os, time, zipfile

ZIP_PATH     = '/content/drive/MyDrive/dissertation/archive.zip'
EXTRACT      = '/content/nih_unzipped'
DATASET_PATH = EXTRACT

os.makedirs(EXTRACT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    total     = len(all_files)

print(f"ZIP contains : {total:,} files")
print(f"Extracting to: {EXTRACT}\n")
t0 = time.time()

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for i, file in enumerate(all_files):
        z.extract(file, EXTRACT)
        pct  = (i + 1) / total * 100
        bar  = '█' * int(pct / 2) + '││' * (50 - int(pct / 2))
        mins = (time.time() - t0) / 60
        print(f"\r[{bar}] {pct:5.1f}% | {i+1:,}/{total:,} | {mins:.1f} min",
              end='', flush=True)

print(f"\n\n✓ Extraction complete in {(time.time()-t0)/60:.1f} minutes")
'''

In [ ]:
DATASET_PATH = '/content/nih_unzipped'

print("=== NIH ChestX-ray14 — Top Level Contents ===\n")
for item in sorted(os.listdir(DATASET_PATH)):
    p = os.path.join(DATASET_PATH, item)
    if os.path.isdir(p):
        print(f"  [DIR]  {item:<30} {len(os.listdir(p))} items")
    else:
        print(f"  [FILE] {item:<30} {os.path.getsize(p)/1e6:.1f} MB")

In [ ]:
print("Building image path index...")
image_path_dict = {}
for folder in sorted(os.listdir(DATASET_PATH)):
    if folder.startswith('images_'):
        img_dir = os.path.join(DATASET_PATH, folder, 'images')
        if os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.endswith('.png'):
                    image_path_dict[f] = os.path.join(img_dir, f)
print(f"✓ Indexed {len(image_path_dict):,} images")
sample = '00000001_000.png'
if sample in image_path_dict:
    print(f"✓ Spot check OK : {image_path_dict[sample]}")

## Section 3 — Metadata Loading, Exploration and Cleaning

In [ ]:
import pandas as pd

df_raw = pd.read_csv(f"{DATASET_PATH}/Data_Entry_2017.csv")
print(f"Rows    : {len(df_raw):,}")
print(f"Columns : {len(df_raw.columns)}")

In [ ]:
import numpy as np

df = df_raw.copy()
df = df.rename(columns={'Patient Gender': 'Patient Sex'})
print("✓ Step 1 — Patient Gender → Patient Sex")

null_cols = [c for c in df.columns if df[c].isnull().all()]
df        = df.drop(columns=null_cols)
print(f"✓ Step 2 — Removed null columns: {null_cols}")

before = len(df)
df     = df[df['Patient Age'] <= 100]
print(f"✓ Step 3 — Removed {before - len(df)} rows (age > 100)")

df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)
print(f"✓ Step 4 — Binary label | Pathology rate: {df['label'].mean()*100:.1f}%")

df['Age Group'] = pd.cut(
    df['Patient Age'],
    bins   = [0, 20, 40, 60, 80, 101],
    labels = ['0-20', '20-40', '40-60', '60-80', '80+']
)
print("✓ Step 5 — Age groups added")

df['image_path'] = df['Image Index'].map(image_path_dict)
print(f"✓ Step 6 — Image paths linked: {df['image_path'].notna().sum():,} matched")
print(f"\nFinal dataset : {len(df):,} rows")

## Section 4 — GPU Optimisation

In [ ]:
import flwr as fl
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
from typing import Dict, List, Optional, Tuple
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
import numpy as np, warnings, json, os, time, copy
warnings.filterwarnings('ignore')

print(f"✓ Flower : {fl.__version__}")
print(f"✓ PyTorch: {torch.__version__}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\n✓ GPU confirmed")

In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")

## Section 5 — Dirichlet Partition → 5 Hospital Clients

In [ ]:
np.random.seed(42)

NUM_CLIENTS    = 5
ALPHA          = 0.5
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

idx_0 = np.where(df['label'].values == 0)[0]
idx_1 = np.where(df['label'].values == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(HOSPITAL_NAMES):
    idx = np.concatenate([
        idx_0[ptr0 : ptr0 + splits_0[i]],
        idx_1[ptr1 : ptr1 + splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy().reset_index(drop=True)
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

print(f"{'Client':<14} {'Images':>8} {'Pathology%':>12}")
print("-" * 36)
for name, data in clients.items():
    print(f"{name:<14} {len(data):>8,} {data['label'].mean()*100:>11.1f}%")
print("\n✓ Non-IID partition confirmed")

## Section 6 — PyTorch Setup: Transforms, Dataset, Model

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std")

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

In [ ]:
SAMPLE_FRACTION = 1.00
TRAIN_SPLIT     = 0.85
VAL_SPLIT       = 0.10
# remaining 0.05 = test

train_clients = {}
val_clients   = {}
test_clients  = {}

for name, data in clients.items():
    data_sample = data.sample(
        frac=SAMPLE_FRACTION, random_state=42
    ).reset_index(drop=True)

    n       = len(data_sample)
    n_train = int(n * TRAIN_SPLIT)
    n_val   = int(n * VAL_SPLIT)

    train_clients[name] = data_sample.iloc[:n_train].reset_index(drop=True)
    val_clients[name]   = data_sample.iloc[n_train:n_train+n_val].reset_index(drop=True)
    test_clients[name]  = data_sample.iloc[n_train+n_val:].reset_index(drop=True)

print(f"=== 10% Dataset | 85/10/5 Split ===\n")
print(f"{'Client':<14} {'Train':>8} {'Val':>6} {'Test':>6} {'Pathology%':>12}")
print("-" * 50)
for name in HOSPITAL_NAMES:
    print(f"{name:<14} {len(train_clients[name]):>8,} "
          f"{len(val_clients[name]):>6,} "
          f"{len(test_clients[name]):>6,} "
          f"{train_clients[name]['label'].mean()*100:>11.1f}%")
print(f"\n✓ train/val/test splits ready — 85/10/5")

## Section 7 — Evaluation Function

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")

## Section 8 — GIFAIR-FL-Per (Yue et al. 2022)

**Algorithm 3 — Yue et al. 2022:**  
Difference from GIFAIR-Global:  
- r_k computed from PERSONALISED losses {theta_k}, not global model  
- Each client retains its own theta_k after each round  
- Server stores one set of personalised weights per hospital  
- Evaluation uses personalised models, not global model  
This prevents extreme clients from destabilising the global model.

In [ ]:
# GIFAIR-Per client — identical to GIFAIR-Global client
# Gradient scaling inside local training is the same
# The difference is handled entirely in the strategy

LAM = 0.1

class GIFAIRPerHospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe
        self.val_dataframe = val_dataframe
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))
        scale  = float(config.get('scale', 1.0))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)

                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()

                # Yue et al. 2022 Algorithm 3 — scale gradients
                for param in self.model.parameters():
                    if param.grad is not None:
                        param.grad.data.mul_(scale)

                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples

        # Also compute personalised loss for r_k update [Yue et al. 2022 Eq.5]
        self.model.eval()
        per_loss_sum, per_n = 0.0, 0
        val_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=val_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False
        )
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                per_loss_sum += criterion(self.model(images), labels).item() * len(labels)
                per_n        += len(labels)
        personalised_loss = per_loss_sum / per_n

        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss'             : float(avg_loss),
             'personalised_loss': float(personalised_loss),
             'client_name'      : self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.val_dataframe), m

print("✓ GIFAIRPerHospitalClient defined — Yue et al. 2022 Algorithm 3")
print("  fit()      → trains on train_clients, computes personalised_loss")
print("  evaluate() → validates on val_clients after each round")

In [ ]:
# GIFAIR-FL-Per strategy — Yue et al. 2022 Algorithm 3
#
# Key differences from GIFAIR-Global:
# 1. Server stores personalised weights theta_k for each hospital
# 2. r_k computed from personalised losses, not global model losses
# 3. After aggregation, server updates theta_k = local weights returned by client
# 4. Evaluation uses personalised theta_k per hospital

class GIFAIRPerStrategy(Strategy):

    def __init__(self, lam: float, num_clients: int, client_sizes: Dict[str, int]):
        super().__init__()
        self.lam         = lam
        self.num_clients = num_clients
        total    = sum(client_sizes.values())
        self.p_k = {name: size / total for name, size in client_sizes.items()}

        # Personalised losses — updated after each round from client's own model
        self.personalised_losses: Dict[str, float] = {n: 1.0 for n in client_sizes}

        # Personalised weights — theta_k for each hospital [Algorithm 3]
        self.personalised_params: Dict[str, Optional[NDArrays]] = \
            {n: None for n in client_sizes}

        self._global_params: Optional[NDArrays] = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        # Initialise personalised params to global model
        for name in self.personalised_params:
            self.personalised_params[name] = [p.copy() for p in ndarrays]
        return ndarrays_to_parameters(ndarrays)

    def _compute_scales(self) -> Dict[str, float]:
        # r_k computed from PERSONALISED losses [Yue et al. 2022 Eq.5]
        # This is the key difference from GIFAIR-Global
        names  = list(self.personalised_losses.keys())
        losses = [self.personalised_losses[n] for n in names]
        scales = {}
        for i, name in enumerate(names):
            r_k = sum(np.sign(losses[i] - losses[j])
                    for j in range(len(losses)) if j != i)
            p_k          = self.p_k[name]
            scales[name] = 1.0 + (self.lam / (p_k * 1)) * r_k
        return scales

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        scales  = self._compute_scales()
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        fit_ins_list = []
        for proxy in sampled:
            cid_int = int(proxy.cid) % NUM_CLIENTS
            name    = HOSPITAL_NAMES[cid_int]
            config  = {'epochs': 3, 'lr': 1e-4,
                       'scale': float(scales.get(name, 1.0)),
                       'server_round': server_round}
            fit_ins_list.append((proxy, FitIns(parameters, config)))
        return fit_ins_list

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        # Aggregation = plain unweighted average (same as GIFAIR-Global)
        all_weights = [parameters_to_ndarrays(r.parameters) for _, r in results]
        n     = len(all_weights)
        w_new = [
            sum(w[layer_idx] for w in all_weights) / n
            for layer_idx in range(len(all_weights[0]))
        ]

        # Update personalised weights and personalised losses [Algorithm 3]
        # Set theta_k = theta_k^{(c+1)E}  — client keeps its own trained weights
        for _, fit_res in results:
            cname = fit_res.metrics.get('client_name', '')
            if cname in self.personalised_params:
                # Store this client's personalised weights
                self.personalised_params[cname] = \
                    parameters_to_ndarrays(fit_res.parameters)
                # Update personalised loss for next round r_k computation
                self.personalised_losses[cname] = float(
                    fit_res.metrics.get('personalised_loss', 1.0))

        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        return [(client, ins) for client in sampled]

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}

        print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
        for _, res in results:
            name = res.metrics.get('client_name', '?')
            auc  = res.metrics.get('auc', float('nan'))
            fnr  = res.metrics.get('fnr', float('nan'))
            print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
        aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
        valid_aucs = [a for a in aucs if not (a != a)]
        if valid_aucs:
            print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                  f"Variance : {float(np.var(valid_aucs)):.6f}")

        total_loss    = sum(r.loss * r.num_examples for _, r in results)
        total_samples = sum(r.num_examples for _, r in results)
        return total_loss / total_samples, {}

    def evaluate(self, server_round, parameters):
        return None

print("✓ GIFAIRPerStrategy defined — Yue et al. 2022 Algorithm 3")
print(f"  λ = {LAM}")
print("  r_k from PERSONALISED losses — key difference from GIFAIR-Global")
print("  theta_k stored per hospital after each round")

In [ ]:
# GIFAIR-Per client factory

def make_gifair_per_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return GIFAIRPerHospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print(f"✓ GIFAIR-Per client factory defined")

In [ ]:
# Run GIFAIR-FL-Per simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 60)
print("GIFAIR-FL-Per — Yue et al. 2022 Algorithm 3")
print(f"Rounds: {ROUNDS} | λ: {LAM} | Clients: {NUM_CLIENTS} | Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 60)

t0 = time.time()

client_sizes        = {name: len(data) for name, data in train_clients.items()}
gifair_per_strategy = GIFAIRPerStrategy(
    lam          = LAM,
    num_clients  = NUM_CLIENTS,
    client_sizes = client_sizes
)

gifair_per_history = fl.simulation.start_simulation(
    client_fn        = make_gifair_per_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = gifair_per_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*60}")
print(f"✓ GIFAIR-FL-Per complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in gifair_per_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

In [ ]:
# Evaluate GIFAIR-FL-Per using PERSONALISED weights per hospital
# This is the key difference from GIFAIR-Global evaluation
# Each hospital is evaluated on its own theta_k, not the global model

import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

gifair_per_metrics = {}

for name, data in test_clients.items():
    # Use personalised weights for this hospital [Algorithm 3 — Return {theta_k}]
    per_params = gifair_per_strategy.personalised_params[name]
    if per_params is None:
        # Fallback to global if personalised not available
        per_params = gifair_per_strategy._global_params

    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in per_params]
    )))
    gifair_per_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = gifair_per_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_gifair_per = pd.DataFrame(rows).set_index('Client')
print("=== GIFAIR-FL-Per Results — Yue et al. 2022 Algorithm 3 ===\n")
print(df_gifair_per.to_string())

valid_aucs = [gifair_per_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (gifair_per_metrics[n]['auc'] != gifair_per_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")
print("Note: evaluated on personalised theta_k per hospital, not global model")

In [ ]:
# Save GIFAIR-FL-Per results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/gifair_per_full_metrics.json', 'w') as f:
    json.dump(gifair_per_metrics, f, indent=2, default=str)

# Save global model
torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in gifair_per_strategy._global_params])),
    f'{SAVE_DIR}/gifair_per_full_global_model.pth'
)

# Save personalised models per hospital
for name in HOSPITAL_NAMES:
    per_params = gifair_per_strategy.personalised_params[name]
    if per_params is not None:
        torch.save(
            dict(zip(build_model().state_dict().keys(),
                     [torch.tensor(v) for v in per_params])),
            f'{SAVE_DIR}/gifair_per_full_{name}_model.pth'
        )

print("✓ GIFAIR-Per metrics saved → gifair_per_full_metrics.json")
print("✓ GIFAIR-Per global model  → gifair_per_full_global_model.pth")
print("✓ GIFAIR-Per personalised models saved per hospital")